# Cross-NBFC Payment Behaviour Analysis
### Identifying Wilful Defaulters via Equifax Credit Bureau Data

---

**Author:** Vinayak  
**Team:** Business Intelligence, Muthoot Capital Services Ltd. (Our Company)  

---

## Objective

This notebook identifies borrowers who hold loans with **both Our Company (ON US) and external NBFCs (OFF US)**, 
and who demonstrate **better repayment behaviour with other lenders than with Our Company**.

Such customers are flagged as **wilful defaulters** and are prioritised for targeted collection,
with the goal of normalising delinquent accounts and reducing the company's **Gross NPA (GNPA)**.

---

## Notebook Structure

| # | Section |
|---|---|
| 1 | Configuration & Imports |
| 2 | Data Loading & Inspection |
| 3 | Data Cleaning & Type Enforcement |
| 4 | Sector Classification (ON US / OFF US) |
| 5 | DPD Bucketing |
| 6 | Multi-Lender Customer Identification |
| 7 | Worst-Case DPD Selection |
| 8 | DPD Cross-Matrix Construction |
| 9 | Results & Interpretation |

---
## 1. Configuration & Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

# ─────────────────────────────────────────────────────────────────
# PROJECT CONFIGURATION
# Update FILE_PATH to point to your local copy of the dataset.
# The actual data is confidential and not included in this repo.
# A synthetic sample dataset is available in data/sample/
# ─────────────────────────────────────────────────────────────────

FILE_PATH       = "data/sample/sample_data.csv"   # Replace with your file path
ON_US_VALUE     = "Muthoot Capital Services Ltd"  # Sector label for Our Company loans
DPD_COLUMN      = "days_past_due"                 # Column name for Days Past Due
SECTOR_COLUMN   = "sector"                        # Column name for lender/sector
REF_COLUMN      = "reference_no"                  # Primary key — unique customer ID

# DPD bucket order — used to maintain consistent matrix ordering
BUCKET_ORDER = ["0/STD", "1-30", "30-60", "60-90", "90+"]
BUCKET_RANK  = {b: i for i, b in enumerate(BUCKET_ORDER)}

print("✅ Configuration loaded.")

---
## 2. Data Loading & Inspection

The source data is an Equifax credit bureau scrub file (~1.3 crore records) originally delivered 
as a pipe-delimited (`|`) flat file. It has been pre-converted to CSV for this notebook.

Key columns retained for analysis:
- `reference_no` — Unique customer identifier (Primary Key)
- `sector` — Lending institution name
- `days_past_due` — EMI Days Past Due at the time of scrub

In [ ]:
df = pd.read_csv(FILE_PATH)

print(f"Dataset shape       : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Total unique customers (reference_no): {df[REF_COLUMN].nunique():,}")
print()
print("Column names:", df.columns.tolist())
print()
print("Data types:")
print(df.dtypes)
print()
print("First 5 rows:")
df.head()

In [ ]:
# ── Missing value summary ──────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count' : missing,
    'Missing (%)' : missing_pct
}).query('`Missing Count` > 0')

print("Missing value summary:")
print(missing_summary if not missing_summary.empty else "No missing values found.")
print()

# ── Sector distribution ────────────────────────────────────────────
print(f"Unique sectors (lenders): {df[SECTOR_COLUMN].nunique():,}")
print()
print(f"Top 10 sectors by record count:")
print(df[SECTOR_COLUMN].value_counts().head(10))

---
## 3. Data Cleaning & Type Enforcement

In [ ]:
initial_count = len(df)

# ── Drop records missing critical fields ───────────────────────────
df = df.dropna(subset=[REF_COLUMN, DPD_COLUMN]).copy()
print(f"Rows dropped (missing reference_no or DPD) : {initial_count - len(df):,}")
print(f"Rows remaining                             : {len(df):,}")

# ── Enforce correct data types ─────────────────────────────────────
df[REF_COLUMN]   = df[REF_COLUMN].astype(str).str.strip()
df[SECTOR_COLUMN] = df[SECTOR_COLUMN].astype(str).str.strip()
df[DPD_COLUMN]   = pd.to_numeric(df[DPD_COLUMN], errors='coerce')

# Drop any rows where DPD became NaN after coercion (non-numeric entries)
coerce_dropped = df[DPD_COLUMN].isna().sum()
df = df.dropna(subset=[DPD_COLUMN]).copy()
print(f"Rows dropped (non-numeric DPD after coerce): {coerce_dropped:,}")

# ── Clip negative DPD values to 0 (treat as standard/current) ─────
df[DPD_COLUMN] = df[DPD_COLUMN].clip(lower=0)

print()
print(f"✅ Clean dataset: {len(df):,} rows | {df[REF_COLUMN].nunique():,} unique customers")

---
## 4. Sector Classification — ON US vs. OFF US

Each loan record is classified as:
- **ON US** → Loan is from Our Company (our company)
- **OFF US** → Loan is from any other external NBFC

In [ ]:
df['loan_type'] = np.where(
    df[SECTOR_COLUMN] == ON_US_VALUE,
    'ON US',
    'OFF US'
)

loan_type_summary = df.groupby('loan_type').agg(
    Total_Records    = (REF_COLUMN, 'count'),
    Unique_Customers = (REF_COLUMN, 'nunique')
)

print("Loan Type Distribution:")
print(loan_type_summary)

on_us  = df[df['loan_type'] == 'ON US'].copy()
off_us = df[df['loan_type'] == 'OFF US'].copy()

---
## 5. DPD Bucketing

Days Past Due (DPD) is bucketed into standard collection categories aligned with 
RBI asset classification norms used in the NBFC sector.

In [ ]:
def assign_dpd_bucket(dpd: float) -> str:
    """
    Assign a DPD bucket label based on days past due value.

    Parameters
    ----------
    dpd : float
        Days past due for a loan record.

    Returns
    -------
    str
        One of: '0/STD', '1-30', '30-60', '60-90', '90+'
    """
    if pd.isna(dpd) or dpd <= 0:
        return "0/STD"
    elif dpd <= 30:
        return "1-30"
    elif dpd <= 60:
        return "30-60"
    elif dpd <= 90:
        return "60-90"
    else:
        return "90+"


df['dpd_bucket']  = df[DPD_COLUMN].apply(assign_dpd_bucket)
df['bucket_rank'] = df['dpd_bucket'].map(BUCKET_RANK)

print("DPD Bucket Distribution (all records):")
print(
    df['dpd_bucket']
    .value_counts()
    .reindex(BUCKET_ORDER)
    .rename('Count')
    .to_frame()
)

# Propagate bucket columns to segment DataFrames
on_us  = df[df['loan_type'] == 'ON US'].copy()
off_us = df[df['loan_type'] == 'OFF US'].copy()

---
## 6. Multi-Lender Customer Identification

We isolate customers who appear in **both** the ON US and OFF US segments — 
i.e., they hold active loans with Our Company **and** at least one external NBFC simultaneously.

In [ ]:
on_us_customers  = set(on_us[REF_COLUMN])
off_us_customers = set(off_us[REF_COLUMN])

both_ids   = on_us_customers & off_us_customers  # Multi-lender: ON US + OFF US
on_only    = on_us_customers - off_us_customers   # Our Company only (no OFF US loan)

print("─" * 45)
print(f"  Unique ON US customers        : {len(on_us_customers):>8,}")
print(f"  Unique OFF US customers       : {len(off_us_customers):>8,}")
print("─" * 45)
print(f"  Our Company-only (no OFF US loan)    : {len(on_only):>8,}  ← 'On member only' row")
print(f"  Multi-lender (ON US + OFF US) : {len(both_ids):>8,}  ← Core analysis target")
print("─" * 45)

# Count OFF US loan count per multi-lender customer
off_us_loan_counts = (
    off_us[off_us[REF_COLUMN].isin(both_ids)]
    .groupby(REF_COLUMN)[SECTOR_COLUMN]
    .nunique()
    .rename('off_us_lender_count')
)

seg_a = (off_us_loan_counts == 1).sum()   # 1 ON US + 1 OFF US
seg_b = (off_us_loan_counts > 1).sum()    # 1 ON US + 2+ OFF US

print(f"  ↳ Segment A (1 ON US + 1 OFF US)    : {seg_a:>6,}")
print(f"  ↳ Segment B (1 ON US + 2+ OFF US)   : {seg_b:>6,}")

---
## 7. Worst-Case DPD Selection

For customers in **Segment B** (multiple OFF US loans), we apply a **conservative worst-case rule**:
the OFF US record with the **highest DPD** is selected as the representative repayment behaviour.
This ensures we do not underestimate the borrower's true credit risk profile.

The same worst-case logic is applied to ON US records to handle edge cases 
where a customer may have multiple Our Company loan accounts.

In [ ]:
def get_worst_bucket(segment_df: pd.DataFrame, bucket_col_name: str) -> pd.DataFrame:
    """
    For each customer, retain the single record with the highest (worst) DPD bucket.

    Parameters
    ----------
    segment_df    : DataFrame filtered to either ON US or OFF US records.
    bucket_col_name : Output column name for the resulting DPD bucket.

    Returns
    -------
    DataFrame with columns [reference_no, bucket_col_name]
    """
    return (
        segment_df
        .sort_values('bucket_rank', ascending=False)
        .groupby(REF_COLUMN, as_index=False)
        .first()
        [[REF_COLUMN, 'dpd_bucket']]
        .rename(columns={'dpd_bucket': bucket_col_name})
    )


on_us_worst  = get_worst_bucket(on_us,  'on_us_bucket')
off_us_worst = get_worst_bucket(off_us, 'off_us_bucket')

print(f"ON US  worst-case records  : {len(on_us_worst):,}")
print(f"OFF US worst-case records  : {len(off_us_worst):,}")
print()
print("Sample — ON US worst DPD per customer:")
print(on_us_worst.head())
print()
print("Sample — OFF US worst DPD per customer:")
print(off_us_worst.head())

---
## 8. DPD Cross-Matrix Construction

The final matrix cross-tabulates:
- **Rows** → OFF US DPD bucket (repayment behaviour with other NBFCs)
- **Columns** → ON US DPD bucket (repayment behaviour with Our Company)

An additional **'On member only'** row captures Our Company customers who have no OFF US loans.

### How to read the matrix:
| Region | Interpretation |
|---|---|
| Low OFF US DPD + High ON US DPD 🔴 | **Wilful defaulters** — paying others, not Our Company |
| High OFF US DPD + High ON US DPD 🟠 | Genuinely stressed borrowers |
| Low OFF US DPD + Low ON US DPD 🟢 | Healthy repayers — monitor only |

In [ ]:
# ── Merge ON US and OFF US worst-case records ──────────────────────
both = pd.merge(on_us_worst, off_us_worst, on=REF_COLUMN, how='inner')
print(f"Multi-lender customers in final merge : {len(both):,}")

# ── Build the cross-matrix ─────────────────────────────────────────
matrix = (
    both
    .groupby(['off_us_bucket', 'on_us_bucket'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=BUCKET_ORDER, columns=BUCKET_ORDER, fill_value=0)
)

# ── 'On member only' row (Our Company customers with no OFF US loan) ──────
on_only_ids    = set(on_us_worst[REF_COLUMN]) - set(off_us_worst[REF_COLUMN])
on_only_counts = (
    on_us_worst[on_us_worst[REF_COLUMN].isin(on_only_ids)]
    ['on_us_bucket']
    .value_counts()
    .reindex(BUCKET_ORDER, fill_value=0)
)
on_member_row = pd.DataFrame(
    [on_only_counts.values],
    columns=BUCKET_ORDER,
    index=["On member only"]
)

# ── Concatenate to produce the final matrix ────────────────────────
final_matrix = pd.concat([on_member_row, matrix])
final_matrix.index.name   = "OFF US  ↓  |  ON US →"
final_matrix.columns.name = None

print()
print("=" * 65)
print("          PAYMENT BEHAVIOUR CROSS-MATRIX")
print("=" * 65)
print(final_matrix.to_string())
print("=" * 65)

---
## 9. Results & Interpretation

In [ ]:
# ── Flag wilful defaulters: OFF US bucket < ON US bucket ──────────
# These customers pay external lenders better than they pay Our Company.

both['on_us_rank']  = both['on_us_bucket'].map(BUCKET_RANK)
both['off_us_rank'] = both['off_us_bucket'].map(BUCKET_RANK)

wilful_defaulters = both[both['on_us_rank'] > both['off_us_rank']].copy()

print("─" * 50)
print(f"  Total multi-lender customers   : {len(both):>8,}")
print(f"  Flagged as wilful defaulters   : {len(wilful_defaulters):>8,}")
print(f"  (Better OFF US than ON US DPD) : {len(wilful_defaulters)/len(both)*100:>7.1f}%")
print("─" * 50)

print()
print("Wilful defaulter breakdown by ON US bucket (collection priority):")
print(
    wilful_defaulters['on_us_bucket']
    .value_counts()
    .reindex(BUCKET_ORDER)
    .dropna()
    .rename('Customer Count')
    .to_frame()
)

print()
print("Sample wilful defaulter records (reference_no masked for privacy):")
display_cols = ['on_us_bucket', 'off_us_bucket']
print(wilful_defaulters[display_cols].head(10).to_string(index=False))

In [ ]:
# ── Export outputs ─────────────────────────────────────────────────

OUTPUT_MATRIX = "outputs/dpd_cross_matrix.csv"
OUTPUT_FLAGS  = "outputs/wilful_defaulters.csv"

final_matrix.to_csv(OUTPUT_MATRIX)

# Export flagged customers — reference_no only (no PII)
wilful_defaulters[[REF_COLUMN, 'on_us_bucket', 'off_us_bucket']].to_csv(
    OUTPUT_FLAGS, index=False
)

print(f"✅ DPD cross-matrix saved   → {OUTPUT_MATRIX}")
print(f"✅ Wilful defaulter list saved → {OUTPUT_FLAGS}")
print()
print("Analysis complete. The flagged customer list is ready for the Collections team.")

---
## Notes

- **Data Privacy:** The actual Equifax scrub dataset is confidential and is not included in this repository. A synthetic sample with the same schema is available in `data/sample/`.
- **Worst-case DPD:** For customers with multiple OFF US loans, only the highest DPD record is used as the representative behaviour — a deliberate conservative choice.
- **Wilful default flag:** A customer is flagged only when their ON US DPD bucket is strictly worse than their OFF US DPD bucket. Equal buckets are excluded from flagging.

---
*For questions or collaboration, connect via [LinkedIn](#).*